# Volatility Dynamics 06c — Student-t emission HMM

Follow-up to **VD 011**: in 06b the diagonal-Gaussian HMM's validation log-likelihood rose monotonically to the edge of the K grid (K = 7). One explanation is emission misspecification: heavy tails force a Gaussian HMM to spend extra states on the tails. 06c replaces the emission with a **multivariate Student-t** (diagonal scale, per-state $\nu_k$, one latent scale per bar shared across fields, missing fields marginalized exactly) and keeps every other part of 06b unchanged.

$$x_t\mid S_t=k \sim t_{\nu_k}(\mu_k,\ \operatorname{diag}(s^2_k))$$

On synthetic heavy-tailed data with true $K=3$, the Gaussian HMM reproduces 06b's pattern (monotone to the grid edge, K = 7) while the Student-t HMM selects K = 3 and recovers $\nu\approx3$.

**Pre-registered (VD 012):**
1. **Primary — identical to 06b:** at $h=60$ min, $\text{logloss}(M3)-\text{logloss}(M1)<0$ with 95% day-bootstrap upper bound $<0$ and ≥ 20 unique held-out onset timestamps.
2. **K:** grid 2–10; *interior K* iff the selected K < 10 and validation LL/row at K = 10 is not above it.
3. **Emission:** Student-t fits better iff its validation LL/row at its selected K exceeds the Gaussian's at the Gaussian's selected K (both fitted here on identical sequences).

The emissions contain no displacement-from-VWAP variable, so no state here can be a displacement (R) regime, whatever K is.

In [ ]:
from pathlib import Path
import json, os, sys, time, warnings
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
from dotenv import load_dotenv

warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 300)
pd.set_option('display.max_rows', 300)

load_dotenv()
DATA_ROOT = Path(os.environ['DERIBIT_DATA_ROOT']).expanduser().resolve()
DB_PATH = DATA_ROOT / 'database' / 'research.duckdb'
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    raise RuntimeError('Repository root not found.')
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from data_engineering.modeling import volatility_vd_b_core as vb
from data_engineering.modeling import volatility_hmm_states_b as hb
from data_engineering.modeling import volatility_hmm_states_c as hc
START = os.getenv('VD6C_START', '').strip() or None
END = os.getenv('VD6C_END', '').strip() or None
ASSETS = [x.strip().upper() for x in os.getenv('VD6C_ASSETS', '').split(',') if x.strip()]
TRAIN_FRACTION = float(os.getenv('VD6C_TRAIN_FRACTION', str(hb.DEFAULT_TRAIN_FRACTION)))
K_VALUES = tuple(int(x) for x in os.getenv('VD6C_K', ','.join(map(str, hc.DEFAULT_K))).split(',') if x.strip())
N_INIT = int(os.getenv('VD6C_N_INIT', str(hc.DEFAULT_N_INIT)))
N_INIT_GAUSS = int(os.getenv('VD6C_N_INIT_GAUSS', '1'))
RUN_GAUSSIAN = os.getenv('VD6C_RUN_GAUSSIAN', '1').strip().lower() not in {'0', 'false', 'no'}
MAX_ITER = int(os.getenv('VD6C_MAX_ITER', str(hc.DEFAULT_MAX_ITER)))
N_BOOT = int(os.getenv('VD6C_N_BOOT', '2000'))
N_SIMS = int(os.getenv('VD6C_N_SIMS', '30'))
OUT = PROJECT_ROOT / 'notebooks' / 'Volatility_Dynamics' / '06c'
OUT.mkdir(parents=True, exist_ok=True)
print('06b pipeline', hb.VERSION, '| 06c', hc.VERSION, hc.selftest())


In [ ]:
def qident(name):
    return '"' + str(name).replace('"', '""') + '"'

def load_frame(columns, start=None, end=None, assets=None):
    cols = list(dict.fromkeys(['state_time', 'underlying', 'return_5m'] + list(columns)))
    where = []
    if start:
        ts = pd.Timestamp(start)
        ts = ts.tz_localize('UTC') if ts.tzinfo is None else ts.tz_convert('UTC')
        where.append(f"state_time >= TIMESTAMPTZ '{ts.isoformat()}'")
    if end:
        ts = pd.Timestamp(end)
        ts = ts.tz_localize('UTC') if ts.tzinfo is None else ts.tz_convert('UTC')
        where.append(f"state_time < TIMESTAMPTZ '{(ts + pd.Timedelta(days=1)).isoformat()}'")
    if assets:
        where.append('underlying IN (' + ','.join(repr(x) for x in assets) + ')')
    sql = ('SELECT ' + ', '.join(qident(c) for c in cols) + ' FROM time_series_feature_state'
           + ('' if not where else ' WHERE ' + ' AND '.join(where)) + ' ORDER BY underlying, state_time')
    con = duckdb.connect(str(DB_PATH), read_only=True)
    try:
        df = con.execute(sql).df()
    finally:
        con.close()
    if df.empty:
        raise RuntimeError('No rows found.')
    df['state_time'] = pd.to_datetime(df['state_time'], utc=True)
    if df.duplicated(['underlying', 'state_time']).any():
        raise RuntimeError('Duplicate asset/timestamp rows.')
    print(f'rows {len(df):,} | assets {sorted(df.underlying.unique())} | {df.state_time.min()} -> {df.state_time.max()}')
    return df.reset_index(drop=True)

def schema_columns():
    con = duckdb.connect(str(DB_PATH), read_only=True)
    try:
        return con.execute('DESCRIBE time_series_feature_state').df()
    finally:
        con.close()

schema = set(schema_columns()['column_name'].astype(str))
missing = set(hb.REQUIRED_COLUMNS) - schema
if missing:
    raise RuntimeError(f'missing columns: {missing}')
df = load_frame(list(hb.REQUIRED_COLUMNS), START, END, ASSETS)
grid = vb.build_grid(df)


In [ ]:
E = hb.build_emissions(grid, include_l2=False)
split = hb.chronological_split(grid, TRAIN_FRACTION)
train_mask = grid.time_ns < split.test_start_ns
params = hb.fit_hour_normalizer(grid, E, train_mask)
Z = hb.apply_normalizer(grid, E, params)
fit_idx = hb.sequences(grid, grid.time_ns < split.valid_start_ns)
val_idx = hb.sequences(grid, (grid.time_ns >= split.valid_start_ns) & train_mask)
tr_idx = hb.sequences(grid, train_mask)
te_idx = hb.sequences(grid, ~train_mask)
seq_meta = pd.DataFrame([{'split': s, 'n_sequences': len(ix), 'rows': int(sum(map(len, ix)))}
                         for s, ix in [('fit', fit_idx), ('valid', val_idx), ('train', tr_idx), ('test', te_idx)]])
display(seq_meta)
print('test starts', pd.Timestamp(split.test_start_ns, tz='UTC'), '| validation starts', pd.Timestamp(split.valid_start_ns, tz='UTC'))


In [ ]:
t0 = time.time()
comparison, model, k_t, k_g = hc.select_and_fit_both(
    [Z[i] for i in fit_idx], [Z[i] for i in val_idx], [Z[i] for i in tr_idx], K_VALUES,
    n_init=N_INIT, n_init_gauss=N_INIT_GAUSS, max_iter=MAX_ITER, run_gaussian=RUN_GAUSSIAN, progress=True)
print(f'fit {time.time()-t0:.0f}s | Student-t selected K = {k_t} | Gaussian selected K = {k_g}')
questions = hc.k_and_emission_questions(comparison, k_t, k_g)
print(json.dumps(questions, indent=2))
display(comparison.pivot(index='K', columns='emission', values='valid_loglik_per_row').round(4))
fig, ax = plt.subplots(figsize=(7, 3))
for em, g in comparison.groupby('emission'):
    ax.plot(g.K, g.valid_loglik_per_row, marker='o', label=em)
ax.set_xlabel('K'); ax.set_ylabel('validation log-lik / row'); ax.legend(); plt.show()
emis = pd.DataFrame(model.means, columns=E.columns).rename_axis('state')
emis['nu'] = model.nu
display(emis.round(2))
display(pd.DataFrame(model.A).round(3))


In [ ]:
ft_train = hb.filtered_table(grid, model, tr_idx, Z, 'train')
ft_test = hb.filtered_table(grid, model, te_idx, Z, 'test')
thr = vb.q99_thresholds(grid, row_mask=train_mask)
tail = vb.tail_flags(grid, thr)
onset = vb.onset_flags(grid, tail)
labels, first_event = hb.future_onset_labels(grid, onset)
bench = hb.benchmark_features(grid)
def assemble(ft):
    t = ft.copy()
    t['underlying'] = [grid.assets[c] for c in grid.asset_code[t.row]]
    t['day'] = grid.day[t.row]
    t = pd.concat([t.reset_index(drop=True), labels.iloc[t.row].reset_index(drop=True), bench.iloc[t.row].reset_index(drop=True)], axis=1)
    return t[np.isfinite(t[list(hb.BENCHMARK_FEATURES)]).all(axis=1)].reset_index(drop=True)
tr_tab, te_tab = assemble(ft_train), assemble(ft_test)
test_onsets = np.flatnonzero(onset & ~train_mask)
n_unique_test = len(np.unique(grid.time_ns[test_onsets]))
print('test rows', len(te_tab), '| unique test onset timestamps', n_unique_test)
ft_test['underlying'] = [grid.assets[c] for c in grid.asset_code[ft_test.row]]
ft_test['state_time'] = pd.to_datetime(grid.time_ns[ft_test.row], utc=True)
display(pd.crosstab(ft_test.state_time.dt.date, ft_test.hard_state, normalize='index').round(2))
display(pd.crosstab(ft_test.underlying, ft_test.hard_state, normalize='index').round(2))


In [ ]:
state_cols = [f'p_state_{k}' for k in range(model.K)]
scores, boot = hb.benchmark_comparison(tr_tab, te_tab, state_cols, n_boot=N_BOOT)
decision = hb.hmm_decision(boot, n_unique_test)
display(scores.round(5)); display(boot.round(5))
print(json.dumps(decision, indent=2))
hazard = hb.state_hazard_with_events(te_tab, model.K, first_event, grid)
display(hazard[hazard.horizon_min == 60].round(4))


In [ ]:
t0 = time.time()
dur_summary, dur_hazard = hb.duration_check(model, [Z[i] for i in te_idx], n_sims=N_SIMS)
print(f'duration simulation {time.time()-t0:.0f}s')
display(dur_summary.round(3))


In [ ]:
params.to_csv(OUT / '06c_hour_normalizer.csv', index=False)
seq_meta.to_csv(OUT / '06c_sequence_metadata.csv', index=False)
comparison.to_csv(OUT / '06c_k_selection_t_vs_gaussian.csv', index=False)
emis.reset_index().to_csv(OUT / '06c_emission_locations_and_nu.csv', index=False)
pd.DataFrame(np.sqrt(model.variances), columns=E.columns).rename_axis('state').reset_index().to_csv(OUT / '06c_emission_scales.csv', index=False)
pd.DataFrame(model.A).rename_axis('from_state').reset_index().to_csv(OUT / '06c_transitions.csv', index=False)
ft_test.to_csv(OUT / '06c_filtered_test.csv', index=False)
scores.to_csv(OUT / '06c_benchmark_scores.csv', index=False)
boot.to_csv(OUT / '06c_benchmark_bootstrap.csv', index=False)
hazard.to_csv(OUT / '06c_state_q99_hazard.csv', index=False)
dur_summary.to_csv(OUT / '06c_duration_summary.csv', index=False)
dur_hazard.to_csv(OUT / '06c_duration_hazard.csv', index=False)
(OUT / '06c_decision.json').write_text(json.dumps({'primary': decision, 'k_and_emission': questions}, indent=2))
meta = {
    'version': hc.VERSION, 'pipeline_version_06b': hb.VERSION, 'module': '06c_student_t_hmm',
    'emission': 'multivariate Student-t, diagonal scale, per-state nu, shared latent scale per bar, missing fields marginalized',
    'emissions': list(E.columns), 'k_grid': list(K_VALUES), 'n_init': N_INIT, 'n_init_gaussian': N_INIT_GAUSS,
    'student_t_selected_K': int(k_t), 'gaussian_selected_K': None if k_g is None else int(k_g),
    'nu_by_state': [float(v) for v in model.nu], 'train_fraction': TRAIN_FRACTION,
    'test_start': str(pd.Timestamp(split.test_start_ns, tz='UTC')), 'validation_start': str(pd.Timestamp(split.valid_start_ns, tz='UTC')),
    'n_unique_test_onset_timestamps': int(n_unique_test), 'decision': decision, 'k_and_emission': questions,
    'first_time': str(df.state_time.min()), 'last_time': str(df.state_time.max()), 'n_rows': int(len(df)),
}
(OUT / '06c_run_metadata.json').write_text(json.dumps(meta, indent=2))
print('wrote', OUT)
